# Bangladesh HR Compensation & Compliance Analytics
Synthetic Bangladesh HR dataset for Excel, Power BI, Looker Studio, Python and SQLite.

> ⚠️ Decision-support demo only. Verify current labour, wage-gazette and tax requirements before operational use.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path.cwd().parent if Path.cwd().name == 'python_sqlite' else Path.cwd()
DATA = BASE / 'data'
DB = BASE / 'python_sqlite' / 'hr_analytics.sqlite'
print(DATA, DB)

In [ ]:
employees = pd.read_csv(DATA / 'employees.csv')
payroll = pd.read_csv(DATA / 'payroll_monthly.csv', parse_dates=['Payroll_Month','Due_Date','Pay_Date'])
attendance = pd.read_csv(DATA / 'attendance_leave_monthly.csv', parse_dates=['Month'])
compliance = pd.read_csv(DATA / 'compliance_register.csv', parse_dates=['Month','Action_Due_Date'])
employees.head()

## Data cleaning checks

In [ ]:
for df in [employees, payroll, attendance, compliance]:
    df.columns = df.columns.str.strip()

quality = {
    'duplicate_employee_ids': int(employees['Employee_ID'].duplicated().sum()),
    'missing_appointment_letters': int((employees['Appointment_Letter'] != 'Yes').sum()),
    'late_payroll_records': int((payroll['Paid_On_Time'] != 'Yes').sum()),
    'ot_rate_exceptions': int((payroll['OT_Rate_Compliant'] != 'Yes').sum()),
    'hours_exceptions': int((attendance['Hours_Compliant'] != 'Yes').sum()),
}
pd.Series(quality, name='count')

## SQLite queries

In [ ]:
with sqlite3.connect(DB) as con:
    monthly = pd.read_sql_query('SELECT * FROM vw_monthly_kpis ORDER BY Payroll_Month', con)
monthly

In [ ]:
monthly['Payroll_Month'] = pd.to_datetime(monthly['Payroll_Month'])
monthly.plot(x='Payroll_Month', y=['Total_Gross_Pay','Total_Net_Pay'], marker='o', figsize=(10,5))
plt.title('Monthly Payroll Trend')
plt.ylabel('BDT')
plt.tight_layout()
plt.show()

In [ ]:
with sqlite3.connect(DB) as con:
    risk = pd.read_sql_query('SELECT * FROM vw_employee_risk_flags WHERE High_Risk_Count > 0 ORDER BY High_Risk_Count DESC', con)
risk.head(20)